## objective of this notebook is to estimate the coverage of specific buildings in downtown seattle
- use tables created in notebook 01_Ingest and available in cmegdemos_catalog.network_enablement_demo_test schema
- Perform spatial joins with ST functions to determine which buildings fall within specific coverage hexagons
- calculate distance between buildings and cell towers
- do analysis on distance to a cell tower and signal strength of the h3 grid that the building falls in
- visualize with a heatmap of coverage for downtown seattle - make sure plot is scrollable/zoomable
- use databricks native ST functions and H3 indexing after converting data appropriately wherever possible
- ultimately we should save another table that contains all of this data for buildings in downtown seattle with the h3 coverage information and distance from nearest tower

prompt to genie code: see instructions in this notebook and get to work, feel free to add an additional markdown below with refined instructions after you create your plan and/or ask any questions to me before starting on the work

## Refined Analysis Plan

**Source tables** (`cmegdemos_catalog.network_enablement_demo_test`):
| Table | Key Columns |
| --- | --- |
| `building_footprints` | building_id, height, geometry(4326) |
| `fcc_bdc_h3_seattle` | h3_res9_id, technology, mindown, minup |
| `cell_towers` | tower_id, carrier, tower_type, cell_id, coverage_radius_m, samples, latitude, longitude, location(4326) |

**Downtown Seattle bbox:** lat \[47.595, 47.625\], lon \[-122.355, -122.325\]

**Steps:**
1. Filter buildings to downtown Seattle using centroid coordinates
2. Assign H3 resolution-9 index to each building centroid (`h3_longlatash3string`)
3. LEFT JOIN with `fcc_bdc_h3_seattle` on `h3_res9_id` to get best download/upload speed and provider count
4. CROSS JOIN with `cell_towers` + `ROW_NUMBER()` to find nearest tower per building (haversine distance in meters)
5. Save final table `downtown_seattle_building_coverage` (~1,565 buildings)
6. Visualize: interactive heatmap of coverage quality (folium, scrollable/zoomable)
7. Analysis: distance-to-tower vs. signal strength patterns

In [0]:
# ---- Parameterized config ----
CATALOG = "cmegdemos_catalog"
SCHEMA  = "network_enablement_demo_test"

# Downtown Seattle bounding box
DT_LAT_MIN, DT_LAT_MAX = 47.595, 47.625
DT_LON_MIN, DT_LON_MAX = -122.355, -122.325

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Catalog  : {CATALOG}")
print(f"Schema   : {SCHEMA}")
print(f"Downtown bbox: lat [{DT_LAT_MIN}, {DT_LAT_MAX}], lon [{DT_LON_MIN}, {DT_LON_MAX}]")

In [0]:
# ---------- Downtown buildings with centroid coordinates and H3 res-9 index ----------
df_downtown = spark.sql(f"""
SELECT 
  b.building_id,
  b.height AS building_height,
  b.geometry,
  ST_X(ST_Centroid(b.geometry)) AS centroid_lon,
  ST_Y(ST_Centroid(b.geometry)) AS centroid_lat,
  h3_longlatash3string(
    ST_X(ST_Centroid(b.geometry)), 
    ST_Y(ST_Centroid(b.geometry)), 
    9
  ) AS h3_res9_id
FROM building_footprints b
WHERE ST_Y(ST_Centroid(b.geometry)) BETWEEN {DT_LAT_MIN} AND {DT_LAT_MAX}
  AND ST_X(ST_Centroid(b.geometry)) BETWEEN {DT_LON_MIN} AND {DT_LON_MAX}
""")
df_downtown.createOrReplaceTempView("downtown_buildings")
print(f"Downtown buildings: {df_downtown.count():,}")
display(df_downtown.limit(5))

In [0]:
# ---------- LEFT JOIN on H3 res-9 ID to get best coverage per building ----------
df_coverage = spark.sql("""
SELECT 
  db.building_id,
  db.building_height,
  ANY_VALUE(db.geometry) AS geometry,
  db.centroid_lon,
  db.centroid_lat,
  db.h3_res9_id,
  MAX(c.mindown) AS best_download_mbps,
  MAX(c.minup)   AS best_upload_mbps,
  COUNT(DISTINCT c.fid) AS provider_count
FROM downtown_buildings db
LEFT JOIN fcc_bdc_h3_seattle c ON db.h3_res9_id = c.h3_res9_id
GROUP BY db.building_id, db.building_height,
         db.centroid_lon, db.centroid_lat, db.h3_res9_id
""")
df_coverage.createOrReplaceTempView("buildings_with_coverage")

# Quick summary
covered = df_coverage.filter("best_download_mbps IS NOT NULL").count()
total = df_coverage.count()
print(f"Buildings with H3 coverage: {covered:,} / {total:,} ({100*covered/total:.1f}%)")
display(df_coverage.limit(5))

In [0]:
# ---------- Cross join with towers, compute haversine distance, keep nearest ----------
df_result = spark.sql("""
WITH distances AS (
  SELECT 
    bc.*,
    t.tower_id   AS nearest_tower_id,
    t.carrier    AS nearest_carrier,
    t.tower_type AS nearest_tower_type,
    t.cell_id    AS nearest_cell_id,
    t.coverage_radius_m AS nearest_coverage_radius_m,
    t.samples    AS nearest_tower_samples,
    -- Haversine distance in meters
    ROUND(
      6371000 * 2 * ASIN(SQRT(
        POWER(SIN(RADIANS(t.latitude - bc.centroid_lat) / 2), 2) +
        COS(RADIANS(bc.centroid_lat)) * COS(RADIANS(t.latitude)) *
        POWER(SIN(RADIANS(t.longitude - bc.centroid_lon) / 2), 2)
      )), 1
    ) AS distance_to_tower_m,
    ROW_NUMBER() OVER (
      PARTITION BY bc.building_id 
      ORDER BY 
        6371000 * 2 * ASIN(SQRT(
          POWER(SIN(RADIANS(t.latitude - bc.centroid_lat) / 2), 2) +
          COS(RADIANS(bc.centroid_lat)) * COS(RADIANS(t.latitude)) *
          POWER(SIN(RADIANS(t.longitude - bc.centroid_lon) / 2), 2)
        ))
    ) AS rn
  FROM buildings_with_coverage bc
  CROSS JOIN cell_towers t
)
SELECT * EXCEPT (rn) FROM distances WHERE rn = 1
""")
df_result.createOrReplaceTempView("downtown_coverage_final")
print(f"Final result: {df_result.count():,} rows")
display(df_result.select(
  "building_id", "building_height", "h3_res9_id", 
  "best_download_mbps", "distance_to_tower_m", 
  "nearest_tower_type", "nearest_carrier"
).limit(10))

In [0]:
# ---------- Save as permanent delta table ----------
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.downtown_seattle_building_coverage AS
SELECT * FROM downtown_coverage_final
""")

count = spark.sql(f"SELECT count(*) FROM {CATALOG}.{SCHEMA}.downtown_seattle_building_coverage").collect()[0][0]
print(f"✓ Saved {CATALOG}.{SCHEMA}.downtown_seattle_building_coverage: {count:,} rows")
spark.sql(f"DESCRIBE {CATALOG}.{SCHEMA}.downtown_seattle_building_coverage").show(truncate=False)

In [0]:
%pip install folium --quiet

In [0]:
import folium
from folium.plugins import HeatMap
import pandas as pd
import numpy as np

# ---------- Collect data for visualization from saved table ----------
pdf = spark.sql(f"""
  SELECT centroid_lat, centroid_lon, building_height,
         best_download_mbps, distance_to_tower_m,
         nearest_tower_type, h3_res9_id
  FROM {CATALOG}.{SCHEMA}.downtown_seattle_building_coverage
""").toPandas()

# ---------- Build interactive map ----------
center_lat = pdf['centroid_lat'].mean()
center_lon = pdf['centroid_lon'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=15, 
               tiles='cartodbpositron', control_scale=True)

# Heatmap layer: weighted by download speed (higher = warmer)
heat_data = pdf.dropna(subset=['best_download_mbps'])
heat_points = heat_data[['centroid_lat', 'centroid_lon', 'best_download_mbps']].values.tolist()
HeatMap(
    heat_points, 
    name='Coverage Heatmap (download Mbps)',
    radius=18, blur=12, max_zoom=18,
    gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'lime', 0.8: 'yellow', 1.0: 'red'}
).add_to(m)

# Tower markers
towers_pdf = spark.sql(f"""
  SELECT tower_id, tower_type, latitude, longitude, coverage_radius_m, samples
  FROM {CATALOG}.{SCHEMA}.cell_towers
  WHERE latitude BETWEEN {DT_LAT_MIN - 0.01} AND {DT_LAT_MAX + 0.01}
    AND longitude BETWEEN {DT_LON_MIN - 0.01} AND {DT_LON_MAX + 0.01}
""").toPandas()

for _, row in towers_pdf.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='red',
        fill=True,
        fill_opacity=0.8,
        popup=f"Tower {int(row['tower_id'])} ({row['tower_type']})<br>Radius: {row['coverage_radius_m']}m<br>Samples: {row['samples']}"
    ).add_to(m)

# Buildings with no coverage highlighted
no_cov = pdf[pdf['best_download_mbps'].isna()]
for _, row in no_cov.iterrows():
    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=3, color='gray', fill=True, fill_opacity=0.5,
        popup=f"No coverage data"
    ).add_to(m)

folium.LayerControl().add_to(m)
displayHTML(m._repr_html_())

In [0]:
# ---------- Summary statistics by tower type ----------
FQN = f"{CATALOG}.{SCHEMA}.downtown_seattle_building_coverage"

spark.sql(f"""
SELECT 
  nearest_tower_type,
  COUNT(*) AS buildings,
  ROUND(AVG(distance_to_tower_m), 1) AS avg_distance_m,
  ROUND(MIN(distance_to_tower_m), 1) AS min_distance_m,
  ROUND(MAX(distance_to_tower_m), 1) AS max_distance_m,
  ROUND(AVG(best_download_mbps), 1) AS avg_download_mbps,
  ROUND(AVG(best_upload_mbps), 1)   AS avg_upload_mbps,
  ROUND(AVG(provider_count), 1) AS avg_providers
FROM {FQN}
WHERE best_download_mbps IS NOT NULL
GROUP BY nearest_tower_type
ORDER BY buildings DESC
""").show(truncate=False)

# ---------- Distance buckets vs coverage quality ----------
df_analysis = spark.sql(f"""
SELECT 
  CASE 
    WHEN distance_to_tower_m < 100 THEN '< 100m'
    WHEN distance_to_tower_m < 250 THEN '100-250m'
    WHEN distance_to_tower_m < 500 THEN '250-500m'
    ELSE '500m+'
  END AS distance_bucket,
  COUNT(*) AS building_count,
  ROUND(AVG(best_download_mbps), 1) AS avg_download_mbps,
  ROUND(AVG(best_upload_mbps), 1)   AS avg_upload_mbps,
  ROUND(AVG(provider_count), 1) AS avg_providers,
  ROUND(AVG(building_height), 1) AS avg_building_height
FROM {FQN}
WHERE best_download_mbps IS NOT NULL
GROUP BY 1
ORDER BY 1
""")
print("Coverage quality by distance to nearest tower:")
display(df_analysis)